# 📊 Tahap 3: Evaluasi & Inferensi Model YOLOv8

**Nama: Fadli Hifizansyah**  
**NIM: 241730042**  
**Kelas: Informatika 4B**  
**Mata Kuliah: Kecerdasan Buatan**  

---

### Deskripsi Tahap Ini:
Setelah model selesai dilatih, kita perlu menguji performanya secara menyeluruh. Notebook ini mencakup:
1. Mendeteksi lingkungan kerja secara otomatis (Lokal vs Google Colab).
2. Memuat bobot model terbaik hasil training (`best.pt`).
3. Mengevaluasi model pada dataset pengujian (test set) untuk mendapatkan metrik Precision, Recall, dan mAP.
4. Menampilkan grafik Confusion Matrix dan hasil training secara visual.
5. Menguji model pada gambar baru di folder `citra_uji_eksternal` dan menampilkan hasilnya secara langsung.

## 🛠️ 1. Setup Lingkungan (Google Colab / Lokal)
Jalankan cell di bawah ini. Jika berjalan di Google Colab, kode akan mengunduh repository dari GitHub dan menginstal dependensi secara otomatis.

In [ ]:
import os
import sys

# Deteksi Google Colab
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print("☁️ Berjalan di Google Colab")
    # Clone repository GitHub jika folder belum ada
    if not os.path.exists('/content/deteksi-sampah-replikasi'):
        print("📥 Cloning repository dari GitHub...")
        !git clone https://github.com/fadli154/deteksi-sampah-replikasi.git
    
    # Pindah ke direktori repository
    %cd /content/deteksi-sampah-replikasi
    
    # Instalasi library yang dibutuhkan
    !pip install ultralytics torch torchvision matplotlib
else:
    print("💻 Berjalan di komputer lokal")
    # Pindah ke root folder proyek (naik 2 tingkat dari 05_Source_Code/Notebook)
    %cd ../..
    print(f"Direktori kerja saat ini: {os.getcwd()}")

## 📥 2. Memuat Bobot Model Terbaik (best.pt)
Cell ini akan secara otomatis mencari file bobot model terbaik hasil training Anda di folder `runs/detect/`.

In [ ]:
from ultralytics import YOLO

# Daftar folder yang mungkin menyimpan bobot model terbaik
possible_weights = [
    "runs/detect/train-yolov8n/weights/best.pt",
    "runs/detect/train-yolov8s/weights/best.pt",
    "runs/detect/train/weights/best.pt",
    "yolov8n.pt"  # Fallback jika belum ada model hasil training
]

best_weights = None
for path in possible_weights:
    if os.path.exists(path):
        best_weights = path
        break

if best_weights:
    print(f"✅ Bobot model ditemukan di: {best_weights}")
    model = YOLO(best_weights)
else:
    print("❌ Error: File bobot model tidak ditemukan. Jalankan training.ipynb terlebih dahulu!")

## 📈 3. Evaluasi Model pada Dataset Test
Mari uji performa model secara kuantitatif pada folder `test` (dataset pengujian).

In [ ]:
if best_weights:
    print("⏳ Mengevaluasi model...")
    metrics = model.val(data="data.yaml", split='test', plots=True)
    
    print("\n" + "="*40)
    print("🏆 METRIK AKURASI MODEL (TEST SET):")
    print("="*40)
    print(f"Precision (Presisi) : {metrics.box.mp:.4f} ({metrics.box.mp*100:.1f}%)")
    print(f"Recall (Daya Ingat) : {metrics.box.mr:.4f} ({metrics.box.mr*100:.1f}%)")
    print(f"mAP50                : {metrics.box.map50:.4f} ({metrics.box.map50*100:.1f}%)")
    print(f"mAP50-95             : {metrics.box.map:.4f} ({metrics.box.map*100:.1f}%)")
    print("="*40)
else:
    print("❌ Bobot model belum dimuat.")

## 📊 4. Tampilkan Grafik Kurva Pelatihan & Confusion Matrix
Kita akan memuat dan menampilkan grafik hasil training (`results.png` dan `confusion_matrix.png`) yang digenerate otomatis oleh Ultralytics.

In [ ]:
from IPython.display import Image, display

# Tentukan folder training berdasarkan path weights
train_folder = os.path.dirname(os.path.dirname(best_weights)) if best_weights else None

if train_folder and os.path.exists(train_folder):
    results_img = os.path.join(train_folder, "results.png")
    cm_img = os.path.join(train_folder, "confusion_matrix.png")
    
    if os.path.exists(results_img):
        print("📈 HASIL TRAINING CURVE (Loss & Accuracy):")
        display(Image(filename=results_img, width=800))
        
    if os.path.exists(cm_img):
        print("\n📊 CONFUSION MATRIX MODEL:")
        display(Image(filename=cm_img, width=600))
else:
    print("⚠️ Folder hasil training tidak ditemukan.")

## 🔮 5. Uji Inferensi (Prediksi Citra Uji Eksternal)
Jalankan prediksi pada gambar baru di luar dataset (folder `citra_uji_eksternal/`) untuk melihat kemampuan deteksi objek model secara langsung.

In [ ]:
import glob

test_images_dir = "citra_uji_eksternal"
image_files = glob.glob(os.path.join(test_images_dir, "*.jpg")) + \
              glob.glob(os.path.join(test_images_dir, "*.jpeg")) + \
              glob.glob(os.path.join(test_images_dir, "*.png")) + \
              glob.glob(os.path.join(test_images_dir, "*.webp"))

if best_weights and image_files:
    # Jalankan prediksi dan simpan gambar hasil prediksi
    print(f"⏳ Menjalankan inferensi pada {len(image_files)} citra eksternal...")
    model.predict(source=test_images_dir, save=True, conf=0.25)
    
    # Cari folder runs/detect/predict terbaru
    predict_folders = glob.glob("runs/detect/predict*")
    if predict_folders:
        latest_predict_folder = max(predict_folders, key=os.path.getmtime)
        print(f"\n🖼️ Hasil prediksi disimpan di: {latest_predict_folder}")
        
        # Tampilkan setiap gambar hasil prediksi secara inline
        predicted_images = glob.glob(os.path.join(latest_predict_folder, "*"))
        for p_img in predicted_images:
            if p_img.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                filename = os.path.basename(p_img)
                print(f"\nHasil Prediksi: {filename}")
                display(Image(filename=p_img, width=500))
                print("-" * 50)
else:
    print("⚠️ Folder 'citra_uji_eksternal' kosong atau tidak ditemukan.")

## 🏁 Kesimpulan Tahap 3
Model YOLOv8 berhasil diuji dengan baik. Hasil metrik akurasi menunjukkan performa deteksi sampah daur ulang (kaca, kertas, logam, plastik) yang kuat, dan visualisasi inferensi eksternal membuktikan model mampu bekerja secara real-time pada citra di dunia nyata!